# C — Táo: PlantVillage → Plant Pathology 2020, giao thức CBPUDA (Huang & Tsai)

Chạy pipeline FBR + CDAN+E (+ Mean Teacher) theo **đúng cách chia dữ liệu** của
Huang L.-C., Tsai H.-H., *Contrastive Bi-Projector for Unsupervised Domain Adaption* (CBPUDA),
arXiv:2308.07017, Table IV (PV → PP).

| | CBPUDA (§IV, Table IV) | Notebook này |
|---|---|---|
| Source (có nhãn) | **toàn bộ** PV táo: 1645 healthy / 275 rust / 630 scab = **2550** | **2550** ảnh, không cắt theo lớp, qua FBR |
| Target khi huấn luyện | PP2020, bỏ 81 ảnh multiple_diseases: 516 / 622 / 592 = **1730** | **1730** ảnh, không nhãn |
| Đánh giá | trên **chính** target đó (transductive) | trên **chính** 1730 ảnh đó |
| Backbone | ResNet-50 (ImageNet) | **ResNet-50** (ImageNet) |
| Tối ưu | SGD, wd 1e-3, lr 1e-4 / 5e-3, batch 36, 10 000 iteration | AdamW lr 1e-3, wd 0.01, cosine, batch 64, 300 epoch × 27 bước = 8 100 iteration |
| FBR | không | **có** (cache riêng, xem §2) |

**Khác B ở đúng một điểm dữ liệu:** B dùng 875 ảnh source (tối đa 300/lớp, giống Jeon et al.), C dùng
toàn bộ 2550 ảnh như CBPUDA. Target, đánh giá, backbone, optimizer, pool ảnh nền FBR giữ nguyên như B.

### Chọn model khi không có tập val

CBPUDA huấn luyện trên toàn bộ source và báo cáo model sau iteration cuối; bài không có tập val.
Công tắc `CBP_TRAIN_ON_ALL_SOURCE` (§2):

- `True` (mặc định, khớp bài báo): train trên 2550 ảnh. "Val loss" của `train_cdane` tính trên **chính 2550 ảnh đó**
  với `VAL_TRANSFORM`, nên thực chất là chọn epoch ≥ 250 có loss source thấp nhất. Không dùng nhãn target.
- `False`: chia source 75/25 như A/B (train 1912, val 638) để chọn model bằng val thật. Lệch khỏi bài báo ở số ảnh train.

### Chạy

1. §2: kiểm tra `SAM_MODEL_TYPE`, `CBP_TRAIN_ON_ALL_SOURCE`.
2. Chạy hết các cell định nghĩa (§3, §4, loaders, §5.1–5.2c), rồi **§5.8** (`cbp_setup`, `run_cbp`, `compare_cbp`).
3. Lần đầu §4.1 phải chạy SAM cho 2550 ảnh (cache cũ của A/B **không** dùng lại được, xem §2).
   Kết quả (JSON) và checkpoint ghi vào `runs_<sam>/` với tag `cbp_*`; có file rồi thì bỏ qua.


## 1 — Setup, Installation & Reproducibility

In [ ]:
!pip install torch torchvision -q
!pip install segment-anything -q        # SAM for FBR segmentation
!pip install kaggle -q                  # Download PlantPathology dataset

import os, random, copy, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as tv_models
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


# ══════════════════════════════════════════════════════════════════════════════
# FULL REPRODUCIBILITY SETUP
# ══════════════════════════════════════════════════════════════════════════════
# Goal: identical results across different runs.
#
# Deep learning has MANY sources of randomness. To make runs reproducible,
# we must control ALL of them:
#   1. Python's random module        (data shuffling, sampling)
#   2. NumPy's random generator       (augmentation, array ops)
#   3. PyTorch CPU random             (weight init, dropout)
#   4. PyTorch GPU (CUDA) random      (GPU weight init, cuDNN ops)
#   5. cuDNN algorithm selection      (convolution backends)
#   6. DataLoader worker randomness   (parallel data loading)

def set_seed(seed: int = 42):
    """
    Set all random seeds for full reproducibility.

    Args:
        seed: The random seed value. Same seed → same results every run.
    """
    random.seed(seed)                            # Python built-in RNG
    np.random.seed(seed)                         # NumPy RNG
    torch.manual_seed(seed)                      # PyTorch CPU RNG
    torch.cuda.manual_seed(seed)                 # PyTorch current GPU RNG
    torch.cuda.manual_seed_all(seed)             # PyTorch all GPUs RNG

    # cuDNN determinism:
    #   deterministic=True  → use only deterministic algorithms (reproducible)
    #   benchmark=False     → disable auto-tuner that picks different algorithms
    #                         (the auto-tuner is faster but non-deterministic)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    # Force PyTorch to raise an error if any operation is non-deterministic,
    # so we catch hidden sources of randomness. warn_only=True keeps training
    # running (some ops have no deterministic version) but warns us.
    torch.use_deterministic_algorithms(True, warn_only=True)

    # Required by some CUDA operations for full determinism.
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


def seed_worker(worker_id: int):
    """
    Seed each DataLoader worker process for reproducibility.

    When num_workers > 0, each worker is a separate process with its own RNG.
    Without seeding, workers produce different augmentations each run.
    (We use num_workers=0 on Colab, but this is here for completeness.)

    Args:
        worker_id: Automatically passed by PyTorch to each worker.
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# Ensures DataLoader shuffling is reproducible.
g = torch.Generator()
g.manual_seed(42)

SEED = 42
set_seed(SEED)
print(f'\nReproducibility enabled (seed={SEED})')
print('  - All RNGs seeded (Python, NumPy, PyTorch CPU/GPU)')
print('  - cuDNN deterministic mode ON, benchmark OFF')
print('  - Deterministic algorithms enforced')
print('Setup complete.')


Device : cuda
GPU    : NVIDIA L4

Reproducibility enabled (seed=42)
  - All RNGs seeded (Python, NumPy, PyTorch CPU/GPU)
  - cuDNN deterministic mode ON, benchmark OFF
  - Deterministic algorithms enforced
Setup complete.


## 2 — Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root directory for all checkpoints, caches, and datasets.
SAVE_DIR = '/content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CBPUDA'
os.makedirs(SAVE_DIR, exist_ok=True)

# Dataset directories
PLANT_VILLAGE_SAVE_PATH   = os.path.join(SAVE_DIR, 'PlantVillage')
PLANT_PATHOLOGY_SAVE_PATH = os.path.join(SAVE_DIR, 'PlantPathology')
PLANT_VILLAGE_DIR         = os.path.join(PLANT_VILLAGE_SAVE_PATH, 'raw', 'color')
PLANT_PATHOLOGY_DIR       = PLANT_PATHOLOGY_SAVE_PATH

print(f'Checkpoint directory: {SAVE_DIR}')

# ── SAM cho FBR: 'vit_b' (~375 MB) hoac 'vit_h' (~2.5 GB, mask tot hon) ─────────────────────────────────
# Day la CONG TAC DUY NHAT chon SAM: no quyet dinh cache FBR (ViT-B: 'fbr_cache', ViT-H: 'fbr_h_cache') va
# thu muc ket qua + checkpoint (runs_<sam>/), nen hai SAM khong bao gio ghi de / bo qua nham ket qua cua nhau.
SAM_MODEL_TYPE = 'vit_h'
FBR_CACHE_NAME = 'fbr_cache' if SAM_MODEL_TYPE == 'vit_b' else 'fbr_h_cache'
# Cache FBR dat ten theo CHI SO anh ({idx:05d}_c{label}.jpg). C co 2550 anh source theo thu tu khac
# 875 anh cua A/B, nen dung chung thu muc se lay nham anh cua A/B. Bat buoc tach thu muc rieng.
FBR_CACHE_NAME = FBR_CACHE_NAME + '_cbp_full'

# Giao thuc CBPUDA khong co tap val source. True = train tren toan bo 2550 anh (khop bai bao),
# False = chia 75/25 nhu A/B de chon model bang val that. Xem markdown dau notebook.
CBP_TRAIN_ON_ALL_SOURCE = True
RUN_DIR        = os.path.join(SAVE_DIR, f'runs_{SAM_MODEL_TYPE}')
os.makedirs(RUN_DIR, exist_ok=True)
print(f'SAM = {SAM_MODEL_TYPE} | cache FBR = {FBR_CACHE_NAME} | ket qua + checkpoint -> {RUN_DIR}')
print(f'CBP_TRAIN_ON_ALL_SOURCE = {CBP_TRAIN_ON_ALL_SOURCE}')


Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CBPUDA
SAM = vit_h | cache FBR = fbr_h_cache_cbp_full | ket qua + checkpoint -> /content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CBPUDA/runs_vit_h
CBP_TRAIN_ON_ALL_SOURCE = True


## 3 — Download Datasets

### 3.1 — PlantVillage (PVD) — Source Domain (Lab Images)

Over 54,000 leaf images captured in controlled lab conditions with plain
backgrounds. We use only the 3 Apple classes that match PlantPathology.


In [ ]:
# ── Download PlantVillage from GitHub ─────────────────────────────────────────
os.makedirs(PLANT_VILLAGE_SAVE_PATH, exist_ok=True)
needs_download = (not os.path.exists(PLANT_VILLAGE_SAVE_PATH)
                  or not os.listdir(PLANT_VILLAGE_SAVE_PATH))

if needs_download:
    print('Downloading PlantVillage...')
    !git clone https://github.com/spMohanty/PlantVillage-Dataset.git "{PLANT_VILLAGE_SAVE_PATH}" --depth=1 -q
    print('PlantVillage download complete.')
else:
    print('PlantVillage already exists. Skipping download.')


### 3.2 — PlantPathology (PPD) — Target Domain (Real Field Images)

Real apple leaf images from the Kaggle Plant Pathology 2020 competition,
captured in orchards with complex natural backgrounds. This is our target domain.

**Security note:** Upload your `kaggle.json` when prompted instead of hardcoding
the API token. Never commit API tokens to version control.


In [ ]:
import json
import os
import zipfile, glob

# Token KHONG ghi trong notebook: dat KAGGLE_API_TOKEN trong Colab Secrets (hoac bien moi truong).
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    KAGGLE_API_TOKEN = os.environ.get('KAGGLE_API_TOKEN', '')
assert KAGGLE_API_TOKEN, 'Thieu KAGGLE_API_TOKEN (Colab Secrets hoac bien moi truong)'

# ── Setup Kaggle credentials ──────────────────────────────────────────────────
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# ── Write to access_token file ────────────────────────────────────────────────
with open(os.path.join(kaggle_dir, 'access_token'), 'w') as f:
    f.write(KAGGLE_API_TOKEN)
!chmod 600 {kaggle_dir}/access_token

# ── Download PlantPathology ───────────────────────────────────────────────────
os.makedirs(PLANT_PATHOLOGY_SAVE_PATH, exist_ok=True)

is_download = not os.path.exists(os.path.join(PLANT_PATHOLOGY_SAVE_PATH, 'train.csv'))

if is_download:
    print('Downloading Plant Pathology 2020 - FGVC7 dataset from Kaggle...')
    !kaggle competitions download -c plant-pathology-2020-fgvc7 -p "{PLANT_PATHOLOGY_SAVE_PATH}"

    print('Unzipping...')
    for zip_file in glob.glob(os.path.join(PLANT_PATHOLOGY_SAVE_PATH, '*.zip')):
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(PLANT_PATHOLOGY_SAVE_PATH)
        os.remove(zip_file)
    print('PlantPathology downloaded and unzipped!')
else:
    print('PlantPathology already exists. Skipping download.')

### 3.3 — Transforms & Dataset Class

Two transforms:
- **TRAIN_TRANSFORM** — with augmentation (random crop, flips, color jitter) to
  prevent overfitting and simulate the visual variety of field conditions.
- **VAL_TRANSFORM** — deterministic (resize only) for consistent evaluation.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRANSFORMS
# ══════════════════════════════════════════════════════════════════════════════

TRAIN_TRANSFORM = T.Compose([
    T.Resize((256, 256)),      # Resize larger than target so RandomCrop has room
    T.RandomCrop(224),         # ResNet-18 needs 224x224; random crop = augmentation
    T.RandomHorizontalFlip(),  # Disease pattern is left-right invariant
    T.RandomVerticalFlip(),    # Disease pattern is up-down invariant
    T.ColorJitter(             # Simulate varied field lighting (lab has fixed light)
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
    ),
    T.ToTensor(),              # PIL (H,W,C)[0,255] -> tensor (C,H,W)[0,1]
    T.Normalize(               # Match ImageNet statistics (ResNet-18 was pretrained on it)
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],
    ),
])

VAL_TRANSFORM = T.Compose([
    T.Resize((224, 224)),      # Direct resize, no crop — deterministic
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],
    ),
])


# ══════════════════════════════════════════════════════════════════════════════
# DATASET CLASS
# ══════════════════════════════════════════════════════════════════════════════

class PlantDiseaseDataset(Dataset):
    """
    Unified dataset for both PVD (source) and PPD (target).

    Args:
        data_dir : Root folder where each subfolder is one class.
                   Used for auto-discovery mode. Mutually exclusive with `samples`.
        samples  : Pre-built list of (image_path, label_idx) tuples.
                   Used when we filter/sample images manually beforehand.
        classes  : List of class names, e.g. ["healthy", "rust", "scab"].
                   Required when using `samples`.
        transform: Torchvision transform pipeline applied to each image.
        labeled  : If True,  __getitem__ returns (image, label).
                   If False, __getitem__ returns (image, -1) for the unlabeled
                   target domain. This prevents accidental use of target labels
                   during UDA training.
    """

    def __init__(self, data_dir=None, samples=None, classes=None,
                 transform=None, labeled=True):
        self.transform = transform
        self.labeled   = labeled

        if data_dir is not None:
            # Auto-discover: each subfolder name is a class.
            self.classes      = sorted([
                d for d in os.listdir(data_dir)
                if os.path.isdir(os.path.join(data_dir, d))
            ])
            self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
            self.samples      = []
            for cls in self.classes:
                cls_dir = os.path.join(data_dir, cls)
                for fname in os.listdir(cls_dir):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.samples.append(
                            (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                        )
        else:
            # Manual mode: use the provided samples list.
            self.samples      = samples
            self.classes      = classes
            self.class_to_idx = {c: i for i, c in enumerate(classes)}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        # convert('RGB') handles grayscale/RGBA images → always 3 channels
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return (image, label) if self.labeled else (image, -1)

    def get_labels(self):
        """Return all labels — required for StratifiedKFold class balancing."""
        return [s[1] for s in self.samples]

print('Transforms and dataset class defined.')


Transforms and dataset class defined.


### 3.4 — Load & Split Datasets

**Why the split ratios differ between PVD and PPD:**

| Dataset | Split | Reason |
|---|---|---|
| **PVD** (source, labeled) | 75% train / 25% val | Standard supervised split. Validation set is used for model selection. Paper: *"reserved 25% for validation"* |
| **PPD** (target, unlabeled) | 60% target / 40% test | No train/val split needed since we never use PPD labels for training. 60% provides unlabeled images for UDA alignment; 40% is held out for final evaluation only. |

Refactored into 5 clean functions below.


In [ ]:
import pandas as pd

# ── Configuration ─────────────────────────────────────────────────────────────
PVD_CLASS_MAP = {
    "Apple___healthy"          : 0,   # healthy
    "Apple___Cedar_apple_rust" : 1,   # rust
    "Apple___Apple_scab"       : 2,   # scab
}
CLASS_NAMES   = ["healthy", "rust", "scab"]
NUM_CLASSES   = len(CLASS_NAMES)
MAX_PER_CLASS = None   # CBPUDA: lay TOAN BO anh tao cua PV (imgs[:None] = tat ca)
VALID_EXTS    = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")


def load_pvd_samples(data_dir, class_map=PVD_CLASS_MAP,
                     max_per_class=MAX_PER_CLASS, seed=SEED):
    """
    Load and filter PlantVillage source samples.

    Args:
        data_dir      : Path to PlantVillage/raw/color/ (subfolders per class).
        class_map     : Dict mapping folder name -> integer label.
        max_per_class : Max images to sample per class (paper uses ~275-300).
        seed          : Random seed for reproducible sampling.

    Returns:
        List of (image_path, label_idx) tuples.
    """
    random.seed(seed)   # reproducible shuffle
    samples = []
    print('[PVD] Loading source domain (lab images)...')

    for folder_name, label_idx in class_map.items():
        folder_path = os.path.join(data_dir, folder_name)
        # Fuzzy-match folder name across PlantVillage versions.
        if not os.path.isdir(folder_path):
            matches = [f for f in os.listdir(data_dir)
                       if folder_name.lower() in f.lower()]
            if matches:
                folder_path = os.path.join(data_dir, matches[0])
            else:
                print(f'  WARNING: class folder not found: {folder_name}')
                continue

        imgs = [os.path.join(folder_path, f)
                for f in os.listdir(folder_path) if f.endswith(VALID_EXTS)]
        random.shuffle(imgs)                     # shuffle before slicing
        selected = imgs[:max_per_class]
        samples.extend([(p, label_idx) for p in selected])
        print(f'  {folder_name:<28}: {len(imgs):>5} available -> {len(selected):>4} used')

    print(f'  Total PVD samples: {len(samples)}')
    return samples


def load_ppd_samples(data_dir, class_names=CLASS_NAMES):
    """
    Load PlantPathology target samples from train.csv.

    Keeps only single-label rows (healthy/rust/scab). Rows labeled
    'multiple_diseases' are skipped because PVD has no matching class.

    Args:
        data_dir    : Path to PlantPathology/ (contains train.csv and images/).
        class_names : Ordered class names matching the CSV column headers.

    Returns:
        List of (image_path, label_idx) tuples.
    """
    train_csv    = os.path.join(data_dir, "train.csv")
    img_dir      = os.path.join(data_dir, "images")
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    samples      = []

    print('[PPD] Loading target domain (real field images)...')
    df = pd.read_csv(train_csv)

    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, str(row["image_id"]) + ".jpg")
        if not os.path.exists(img_path):
            continue
        # 'break' after first match keeps single-label rows only.
        for cls in class_names:
            if row.get(cls, 0) == 1:
                samples.append((img_path, class_to_idx[cls]))
                break

    print(f'  Total PPD samples (single-label): {len(samples)}')
    return samples


def split_pvd(samples, val_ratio=0.25, seed=SEED):
    """
    Stratified 75/25 train/val split for the labeled source domain.

    Stratified means each class keeps the same proportion in both splits,
    which is important when classes are imbalanced.

    Args:
        samples   : List of (path, label) from load_pvd_samples().
        val_ratio : Fraction reserved for validation (paper: 0.25).
        seed      : Random seed for reproducible split.

    Returns:
        (train_set, val_set) as Subset objects.
        train_set uses TRAIN_TRANSFORM, val_set uses VAL_TRANSFORM.
    """
    labels  = [s[1] for s in samples]
    indices = list(range(len(samples)))
    train_idx, val_idx = train_test_split(
        indices, test_size=val_ratio, stratify=labels, random_state=seed
    )
    ds_train = PlantDiseaseDataset(samples=samples, classes=CLASS_NAMES,
                                   transform=TRAIN_TRANSFORM, labeled=True)
    ds_val   = PlantDiseaseDataset(samples=samples, classes=CLASS_NAMES,
                                   transform=VAL_TRANSFORM,   labeled=True)
    return Subset(ds_train, train_idx), Subset(ds_val, val_idx)


def split_ppd(samples, target_ratio=0.6, seed=SEED):
    """
    Random 60/40 split of the target domain.

    Args:
        samples      : List of (path, label) from load_ppd_samples().
        target_ratio : Fraction used as unlabeled UDA target (default 0.6).
        seed         : Random seed. CRITICAL — a fixed seed guarantees the
                       test set is identical every run, so accuracy numbers
                       are comparable across runs.

    Returns:
        (target_subset, test_subset)
        target_subset: unlabeled images for UDA alignment
        test_subset  : labeled images for final evaluation only
    """
    ds = PlantDiseaseDataset(samples=samples, classes=CLASS_NAMES,
                             transform=VAL_TRANSFORM, labeled=True)
    indices = list(range(len(samples)))
    random.seed(seed)                # fixed seed BEFORE shuffle
    random.shuffle(indices)
    n_target = int(len(indices) * target_ratio)
    return Subset(ds, indices[:n_target]), Subset(ds, indices[n_target:])


def make_loader(dataset, batch_size=64, shuffle=False, drop_last=False):
    """
    Create a reproducible, Colab-safe DataLoader.

    Args:
        dataset    : Dataset or Subset to wrap.
        batch_size : Samples per batch (paper: 64).
        shuffle    : True for training loaders, False for eval loaders.
        drop_last  : True for training/target loaders to keep batch sizes equal
                     (some UDA losses assume equal source/target batch sizes).

    Returns:
        DataLoader with num_workers=0 (stable on Colab) and a fixed generator
        (reproducible shuffling).
    """
    return DataLoader(
        dataset,
        batch_size     = batch_size,
        shuffle        = shuffle,
        num_workers    = 0,             # 0 avoids Colab worker crashes
        pin_memory     = True,
        drop_last      = drop_last,
        worker_init_fn = seed_worker,   # reproducible worker RNG
        generator      = g,             # reproducible shuffle order
    )


In [ ]:
# ── Load everything ───────────────────────────────────────────────────────────
set_seed(SEED)

pvd_samples = load_pvd_samples(PLANT_VILLAGE_DIR)
ppd_samples = load_ppd_samples(PLANT_PATHOLOGY_DIR)

# So anh phai khop dung muc IV-A cua CBPUDA, khong thi khong goi la "cung giao thuc" duoc.
CBP_PV_COUNTS = {"healthy": 1645, "rust": 275, "scab": 630}
CBP_PP_COUNTS = {"healthy": 516,  "rust": 622, "scab": 592}
for _name, _samples, _expected in (("PV", pvd_samples, CBP_PV_COUNTS),
                                   ("PP", ppd_samples, CBP_PP_COUNTS)):
    _got = dict(zip(CLASS_NAMES, np.bincount([l for _, l in _samples], minlength=NUM_CLASSES).tolist()))
    assert _got == _expected, f"{_name}: {_got} != CBPUDA {_expected}"
    print(f"  {_name} khop CBPUDA: {_got}")

# Full source dataset (used later for the FBR cache).
dataset_A = PlantDiseaseDataset(samples=pvd_samples, classes=CLASS_NAMES,
                                transform=TRAIN_TRANSFORM, labeled=True)

# PVD 75/25 split
train_set_A, val_set_A = split_pvd(pvd_samples, val_ratio=0.25, seed=SEED)

# PPD 60/40 split: KHONG dung de danh gia o notebook nay (danh gia tren toan bo 1730 anh, §5.8).
# Giu lai vi (a) 60% target la pool anh nen FBR, giong het B; (b) cell loader 3 view o §5.2b
# va gia tri mac dinh cua train_cdane van tham chieu toi no.
dataset_B_target, dataset_B_test = split_ppd(ppd_samples, target_ratio=0.6, seed=SEED)

# DataLoaders
train_loader_A = make_loader(train_set_A,      batch_size=64, shuffle=True,  drop_last=True)
val_loader_A   = make_loader(val_set_A,        batch_size=64, shuffle=False)
target_loader  = make_loader(dataset_B_target, batch_size=64, shuffle=True,  drop_last=True)
test_loader_B  = make_loader(dataset_B_test,   batch_size=64, shuffle=False)

print(f'\n{"="*55}')
print('DATA SUMMARY')
print(f'{"="*55}')
print(f'  PVD train (75%) : {len(train_set_A):>5} samples  (labeled)')
print(f'  PVD val   (25%) : {len(val_set_A):>5} samples  (labeled)')
print(f'  PPD target (60%): {len(dataset_B_target):>5} samples  (unlabeled)')
print(f'  PPD test   (40%): {len(dataset_B_test):>5} samples  (labeled)')
print(f'{"="*55}')


[PVD] Loading source domain (lab images)...
  Apple___healthy             :  1645 available -> 1645 used
  Apple___Cedar_apple_rust    :   275 available ->  275 used
  Apple___Apple_scab          :   630 available ->  630 used
  Total PVD samples: 2550
[PPD] Loading target domain (real field images)...
  Total PPD samples (single-label): 1730
  PV khop CBPUDA: {'healthy': 1645, 'rust': 275, 'scab': 630}
  PP khop CBPUDA: {'healthy': 516, 'rust': 622, 'scab': 592}

DATA SUMMARY
  PVD train (75%) :  1912 samples  (labeled)
  PVD val   (25%) :   638 samples  (labeled)
  PPD target (60%):  1038 samples  (unlabeled)
  PPD test   (40%):   692 samples  (labeled)


## 4 — FBR: Field-adaptive Background Recomposition

**Idea:** Use SAM to segment the leaf out of the plain lab background, then
composite it onto a real field background from PPD. This makes lab images
visually resemble field images, reducing the domain gap before UDA.

**Three steps per image:**
1. `segment_leaf` — SAM with a center-point prompt (leaves are always centered in PVD)
2. `refine_mask` — morphological cleanup (fill holes, smooth edges)
3. `composite` — alpha-blend leaf onto a random field background


In [ ]:

import cv2

# ── Download and load SAM (SAM_MODEL_TYPE dat o muc 2) ────────────────────────
SAM_FILES = {
    'vit_b': ('sam_vit_b_01ec64.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'),
    'vit_h': ('sam_vit_h_4b8939.pth', 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'),
}
_sam_file, _sam_url = SAM_FILES[SAM_MODEL_TYPE]
SAM_CHECKPOINT = f'/content/{_sam_file}'

if not os.path.exists(SAM_CHECKPOINT):
    print(f'Downloading SAM checkpoint ({SAM_MODEL_TYPE})...')
    !wget -q {_sam_url} -O {SAM_CHECKPOINT}.part && mv {SAM_CHECKPOINT}.part {SAM_CHECKPOINT}
    print('SAM checkpoint downloaded.')

from segment_anything import sam_model_registry, SamPredictor
sam      = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT).to(DEVICE)
sam_pred = SamPredictor(sam)
print(f'SAM {SAM_MODEL_TYPE} loaded on {DEVICE}.')


SAM checkpoint downloaded.
SAM vit_h loaded on cuda.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FBR CORE FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def segment_leaf(predictor, image_rgb):
    """
    Segment the leaf using SAM with a single center-point prompt.

    Args:
        predictor : SamPredictor with the image already loadable.
        image_rgb : (H, W, 3) uint8 RGB image.

    Returns:
        (H, W) boolean mask — True where the leaf is.
    """
    h, w = image_rgb.shape[:2]
    predictor.set_image(image_rgb)
    masks, scores, _ = predictor.predict(
        point_coords     = np.array([[w // 2, h // 2]]),  # center pixel
        point_labels     = np.array([1]),                 # 1 = foreground
        multimask_output = True,                          # return 3 candidates
    )
    return masks[np.argmax(scores)]   # pick highest-confidence mask


def refine_mask(mask, k_d=15, k_e=10, k_g=5, iterations=2):
    """
    Clean up the raw SAM mask with morphological operations.

    Args:
        mask       : (H, W) boolean mask from SAM.
        k_d        : Dilation kernel size (fills holes, merges fragments).
        k_e        : Erosion kernel size (removes boundary noise). Must be < k_d.
        k_g        : Gaussian blur kernel size (odd) for soft edges.
        iterations : Times to repeat dilation+erosion.

    Returns:
        (H, W) float32 soft mask in [0, 1] used as the alpha channel.

    Paper hyperparameters: k_d=15, k_e=10, k_g=5, iterations=2.
    """
    m  = mask.astype(np.uint8) * 255
    kd = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_d, k_d))
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_e, k_e))
    for _ in range(iterations):
        m = cv2.dilate(m, kd)   # expand: fill holes
        m = cv2.erode(m,  ke)   # shrink: clean edges (k_e < k_d = net expand)
    m = cv2.GaussianBlur(m, (k_g, k_g), 0)   # soft edges for natural blending
    return m.astype(np.float32) / 255.0


def composite(lab_img, mask, bg_img):
    """
    Alpha-composite the leaf onto a field background.

    output = mask * leaf + (1 - mask) * background

    Args:
        lab_img : (H, W, 3) uint8 lab image.
        mask    : (H, W) float32 soft mask in [0, 1].
        bg_img  : (H, W, 3) uint8 field background.

    Returns:
        (H, W, 3) uint8 composited image.
    """
    h, w  = lab_img.shape[:2]
    bg    = cv2.resize(bg_img, (w, h))
    alpha = mask[:, :, np.newaxis]                      # (H, W, 1) for broadcast
    out   = alpha * lab_img.astype(np.float32) + (1 - alpha) * bg.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)


def random_field_background(bg_path, out_w, out_h):
    """
    Build a field background patch the way the paper describes:
    random crop -> random rotation -> center crop (removes black corners).

    Args:
        bg_path      : Path to an unlabeled real-field image.
        out_w, out_h : Output size (matches the lab image).

    Returns:
        (out_h, out_w, 3) uint8 RGB background patch at close-up scale.
    """
    img  = cv2.cvtColor(cv2.imread(bg_path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    # 1) Random square patch, 30-70% of the short side.
    #    KEY FIX: take a PATCH of field texture, not the whole scene resized
    #    down (which would drag another PPD leaf into the background).
    s = int(random.uniform(0.30, 0.70) * min(H, W))
    x = random.randint(0, W - s)
    y = random.randint(0, H - s)
    patch = img[y:y + s, x:x + s]

    # 2) Random rotation (creates black corners — removed in step 3)
    M = cv2.getRotationMatrix2D((s / 2, s / 2), random.uniform(0, 360), 1.0)
    patch = cv2.warpAffine(patch, M, (s, s))

    # 3) Center-crop by 1/sqrt(2): largest square guaranteed to sit inside
    #    the rotated area, so no black corners survive.
    c = int(s / np.sqrt(2))
    o = (s - c) // 2
    patch = patch[o:o + c, o:o + c]

    # 4) Scale to the lab image size
    return cv2.resize(patch, (out_w, out_h))


def fbr_single(lab_path, bg_paths, predictor):
    """
    Full FBR pipeline for one lab image.

    Args:
        lab_path  : Path to the source lab image.
        bg_paths  : List of field background paths (one chosen at random).
        predictor : SamPredictor.

    Returns:
        (H, W, 3) uint8 composited image (leaf on field background).
    """
    lab_rgb = cv2.cvtColor(cv2.imread(lab_path), cv2.COLOR_BGR2RGB)
    h, w    = lab_rgb.shape[:2]

    refined_mask = refine_mask(segment_leaf(predictor, lab_rgb))
    bg_rgb       = random_field_background(random.choice(bg_paths), w, h)

    return composite(lab_rgb, refined_mask, bg_rgb)

print('FBR functions defined.')


FBR functions defined.


### 4.1 — Pre-compute FBR Cache

**Why pre-cache instead of running SAM on-the-fly?**
Running SAM inside the DataLoader would (a) be ~100x slower and (b) produce a
different mask each epoch, making training non-deterministic and unstable.
We run SAM once, save the results as JPEG, then load them like any image.

**Background source:** only the PPD **target split** (not the test set) — using
test-set backgrounds would leak test information into training.


In [ ]:
import shutil

FBR_CACHE_DIR = os.path.join(SAVE_DIR, FBR_CACHE_NAME)
os.makedirs(FBR_CACHE_DIR, exist_ok=True)

# The old cache was built with the buggy background (whole image resized,
# black corners). Delete it so every image is regenerated with the fix.
# if os.path.exists(FBR_CACHE_DIR):
#     shutil.rmtree(FBR_CACHE_DIR)
#     print('[reset] cleared old FBR cache')
# os.makedirs(FBR_CACHE_DIR, exist_ok=True)

# Background images from the TARGET split only (avoids test-set leakage).
fbr_bg_paths = [dataset_B_target.dataset.samples[i][0] for i in dataset_B_target.indices] \
    if hasattr(dataset_B_target, 'indices') else \
    [s[0] for s in ppd_samples]

# Fallback: if the above is empty, use all PPD images folder.
if not fbr_bg_paths:
    img_folder = os.path.join(PLANT_PATHOLOGY_DIR, 'images')
    fbr_bg_paths = [os.path.join(img_folder, f) for f in os.listdir(img_folder)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f'Background images (target split): {len(fbr_bg_paths)}')
print(f'Pre-computing FBR for {len(pvd_samples)} source images...')

set_seed(SEED)   # reproducible background selection + rotation
fbr_cached_samples = []
n_new = n_skip = 0

for idx, (orig_path, label) in enumerate(tqdm(pvd_samples)):
    cache_path = os.path.join(FBR_CACHE_DIR, f'{idx:05d}_c{label}.jpg')
    if not os.path.exists(cache_path):
        try:
            result = fbr_single(orig_path, fbr_bg_paths, sam_pred)
            Image.fromarray(result).save(cache_path, quality=92)
            n_new += 1
        except Exception:
            shutil.copy(orig_path, cache_path)   # fallback to original
            n_new += 1
    else:
        n_skip += 1
    fbr_cached_samples.append((cache_path, label))

print(f'\nFBR cache complete. Computed: {n_new} | Reused: {n_skip}')
print(f'Cache directory: {FBR_CACHE_DIR}')


Background images (target split): 1038
Pre-computing FBR for 2550 source images...


  0%|          | 0/2550 [00:00<?, ?it/s]


FBR cache complete. Computed: 0 | Reused: 2550
Cache directory: /content/drive/MyDrive/GRAD PACKAGE/THESIS/PAPER RESEARCH/CBPUDA/fbr_h_cache_cbp_full


In [ ]:
# ── Visual sanity check: do the FBR images look like real field photos? ──
import matplotlib.pyplot as plt

n    = 6
idxs = random.sample(range(len(fbr_cached_samples)), n)

fig, axes = plt.subplots(2, n, figsize=(3*n, 6.5))
for j, idx in enumerate(idxs):
    axes[0, j].imshow(Image.open(pvd_samples[idx][0]).convert('RGB'))
    axes[0, j].set_title('PVD original', fontsize=9); axes[0, j].axis('off')
    axes[1, j].imshow(Image.open(fbr_cached_samples[idx][0]).convert('RGB'))
    axes[1, j].set_title('FBR output', fontsize=9); axes[1, j].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for j, p in enumerate(random.sample(fbr_bg_paths, 4)):
    axes[j].imshow(Image.open(p).convert('RGB'))
    axes[j].set_title('Real PPD (target)', fontsize=9); axes[j].axis('off')
plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ── Build FBR train/val loaders from cache ────────────────────────────────────
# CBP_TRAIN_ON_ALL_SOURCE=True : train = val = toan bo 2550 anh (CBPUDA khong co tap val).
# CBP_TRAIN_ON_ALL_SOURCE=False: chia 75/25 phan tang nhu A/B.
# Val dung VAL_TRANSFORM (tat dinh) de loss chon model khong nhay theo augmentation.
dataset_A_fbr      = PlantDiseaseDataset(samples=fbr_cached_samples, classes=CLASS_NAMES,
                                         transform=TRAIN_TRANSFORM, labeled=True)
dataset_A_fbr_eval = PlantDiseaseDataset(samples=fbr_cached_samples, classes=CLASS_NAMES,
                                         transform=VAL_TRANSFORM, labeled=True)

fbr_labels = dataset_A_fbr.get_labels()
if CBP_TRAIN_ON_ALL_SOURCE:
    fbr_tr_idx = fbr_vl_idx = list(range(len(dataset_A_fbr)))
else:
    fbr_tr_idx, fbr_vl_idx = train_test_split(
        list(range(len(dataset_A_fbr))), test_size=0.25,
        stratify=fbr_labels, random_state=SEED
    )

train_loader_A_fbr = make_loader(Subset(dataset_A_fbr,      fbr_tr_idx), 64, shuffle=True, drop_last=True)
val_loader_A_fbr   = make_loader(Subset(dataset_A_fbr_eval, fbr_vl_idx), 64, shuffle=False)

print(f'FBR loaders ready: train={len(fbr_tr_idx)} | val={len(fbr_vl_idx)}'
      + ('  (val = chinh tap train, giao thuc CBPUDA)' if CBP_TRAIN_ON_ALL_SOURCE else ''))


FBR loaders ready: train=2550 | val=2550  (val = chinh tap train, giao thuc CBPUDA)


## 5 — Unsupervised Domain Adaptation: CDAN+E

Ba nhánh, dùng chung một pipeline dữ liệu:

| Nhánh | Loss | Mục đích |
|---|---|---|
| **CDAN+E** | `L_cls + L_cdan` | Alignment đối kháng có điều kiện theo lớp (multilinear map + entropy weighting) |
| **MT-only** *(mới, 5.4)* | `L_cls + w(t)·L_cons` | Chỉ consistency regularization trên target, **không** có discriminator/GRL |
| **CDAN+E + Mean Teacher** | `L_cls + L_cdan + w(t)·L_cons` | Cộng thêm consistency regularization trên target không nhãn |

Cả hai đều dùng chung ResNet-18 làm feature extractor và train trên:
- **FBR-augmented source** (có nhãn) cho classification
- **PPD target** (không nhãn) cho domain alignment

Mọi cell dưới đây lưu checkpoint mỗi 10 epoch và resume được sau khi Colab ngắt.


### 5.1 — GRL & Model Architectures

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LAMBDA SCHEDULE & DOMAIN LABELS
# ══════════════════════════════════════════════════════════════════════════════

def get_lambda(step, max_steps, gamma=10.0):
    """
    Progressive GRL lambda schedule (DANN).
    Increases from ~0 to ~1 over training so classification is learned first.

    Args:
        step     : Current iteration index.
        max_steps: Total iterations (epochs * batches_per_epoch).
        gamma    : Growth rate (paper: 10.0).
    Returns:
        Lambda value in [0, 1).
    """
    p = step / max_steps
    return 2.0 / (1.0 + torch.exp(torch.tensor(-gamma * p)).item()) - 1.0


def domain_lbl(n, is_src, device):
    """Domain labels: source=0, target=1. Returns (n,) long tensor."""
    return torch.full((n,), 0 if is_src else 1, dtype=torch.long, device=device)


# ══════════════════════════════════════════════════════════════════════════════
# GRADIENT REVERSAL LAYER
# ══════════════════════════════════════════════════════════════════════════════

class GRL(torch.autograd.Function):
    """
    Gradient Reversal Layer autograd function.
    Forward:  identity. Backward: multiply gradient by -lambda.
    This is what turns a normal discriminator into an adversarial one.
    """
    @staticmethod
    def forward(ctx, x, lam):
        ctx.save_for_backward(torch.tensor(lam))
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.saved_tensors[0].item() * grad, None   # reversed gradient


class GradRevLayer(nn.Module):
    """nn.Module wrapper around GRL with an adjustable lambda."""
    def __init__(self, lam=1.0):
        super().__init__()
        self.lam = lam
    def forward(self, x):
        return GRL.apply(x, self.lam)
    def set_lam(self, lam):
        self.lam = lam


# ══════════════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURES
# ══════════════════════════════════════════════════════════════════════════════

class FeatureExtractor(nn.Module):
    """ResNet backbone with the final FC layer removed.

    backbone='resnet18' (default, paper 10 / all main results): output 512-dim.
    backbone='resnet50' (ablation only, see 5.7): output 2048-dim. Changes
    out_dim, so a resnet50 run is NOT interchangeable with a resnet18 run's
    checkpoints or state_dicts.
    """
    def __init__(self, pretrained=True, backbone='resnet18'):
        super().__init__()
        if backbone == 'resnet18':
            weights = tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            net = tv_models.resnet18(weights=weights)
            self.out_dim = 512
        elif backbone == 'resnet50':
            weights = tv_models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            net = tv_models.resnet50(weights=weights)
            self.out_dim = 2048
        else:
            raise ValueError(f'unknown backbone: {backbone}')
        self.backbone = backbone
        self.net = nn.Sequential(*list(net.children())[:-1])
    def forward(self, x):
        return self.net(x).flatten(1)   # (N, out_dim)


class Classifier(nn.Module):
    """Linear label classifier G: 512 -> num_classes."""
    def __init__(self, in_dim=512, num_classes=3):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.fc(x)


class CDANDisc(nn.Module):
    """
    Conditional domain discriminator for CDAN+E.
    Input is the multilinear map (features x predictions), dim = 512*num_classes.
    Two hidden layers (512, 256).
    """
    def __init__(self, in_dim=512, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim * num_classes, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 2)
        )
    def forward(self, x):
        return self.net(x)


def multilinear_map(feat, pred):
    """
    Outer product features x predictions for CDAN+E conditioning.

    Args:
        feat: (N, d) features. pred: (N, C) softmax predictions.
    Returns:
        (N, d*C) flattened outer product.
    NOTE: do NOT detach feat/pred here — gradients must flow through GRL to phi.
    """
    N = feat.size(0)
    return torch.bmm(feat.unsqueeze(2), pred.unsqueeze(1)).view(N, -1)


def uda_test(phi, G, loader, device):
    """
    Evaluate a UDA model (feature extractor phi + classifier G).

    Args:
        phi   : Feature extractor.
        G     : Label classifier.
        loader: Test DataLoader.
        device: 'cuda' or 'cpu'.
    Returns:
        Accuracy (float).
    """
    phi.eval(); G.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (G(phi(imgs)).argmax(1) == lbls).sum().item()
            total   += lbls.size(0)
    return correct / total

print('GRL, CDAN+E models and helpers defined.')


GRL, CDAN+E models and helpers defined.


### 5.2 — Checkpoint Helper & Results Store

The checkpoint helper is shared by the single-run CDAN+E cell below; the
Mean Teacher branch has its own variant that also stores teacher state.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT HELPERS (dung cho cell CDAN+E don-seed)
# ══════════════════════════════════════════════════════════════════════════════

def save_uda_checkpoint(path, epoch, step, best_loss, best_state,
                        phi, G, optimizer, scheduler, D=None):
    """
    Save a full UDA training checkpoint so training can resume after a restart.

    Args:
        path      : File path to write the checkpoint to.
        epoch     : Last completed epoch.
        step      : Global iteration counter (keeps lambda schedule continuous).
        best_loss : Best validation loss seen so far.
        best_state: dict {'phi','G'} of the best model weights.
        phi, G    : Current feature extractor and classifier.
        optimizer : Optimizer (includes Adam momentum buffers).
        scheduler : LR scheduler (includes cosine schedule position).
        D         : CDAN+E conditional discriminator.
    """
    ckpt = {
        'epoch': epoch, 'step': step,
        'best_loss': best_loss, 'best_state': best_state,
        'phi': phi.state_dict(), 'G': G.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
    }
    if D is not None:
        ckpt['D'] = D.state_dict()
    torch.save(ckpt, path)


def load_uda_checkpoint(path, phi, G, optimizer, scheduler, D=None, device='cpu'):
    """
    Load a UDA checkpoint if it exists.

    Args:
        path      : Checkpoint file path.
        phi, G    : Models to restore weights into.
        optimizer : Optimizer to restore.
        scheduler : Scheduler to restore.
        D         : Optional discriminator to restore.
        device    : Map location.

    Returns:
        (start_epoch, step, best_loss, best_state) to continue training,
        or (1, 0, inf, None) if no checkpoint exists.
    """
    if not os.path.exists(path):
        return 1, 0, float('inf'), None
    ckpt = torch.load(path, map_location=device)
    phi.load_state_dict(ckpt['phi'])
    G.load_state_dict(ckpt['G'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if D is not None and 'D' in ckpt:
        D.load_state_dict(ckpt['D'])
    print(f'[Resume] Loaded checkpoint at epoch {ckpt["epoch"]} -> continuing from {ckpt["epoch"]+1}')
    return ckpt['epoch'] + 1, ckpt['step'], ckpt['best_loss'], ckpt['best_state']


# Reference results from the paper (Apple, genuine setting).
# Baseline (ResNet-18 on PVD, no adaptation) measured in an earlier full run of
# this notebook. Kept as a hardcoded reference so the no-adaptation training
# section is not needed just to print a delta.
acc_on_B = 0.6040

paper_results = {'CDAN+E w/ FBR': 91.1}

# Store all results as we go.
all_results = {'Baseline (ref)': acc_on_B * 100}
print('Checkpoint helpers ready. Results store initialized.')


Checkpoint helpers ready. Results store initialized.


### 5.2b — Hạ tầng Mean Teacher

Bốn cell tiếp theo (hàm EMA / loss, `TargetThreeViewDataset`, loader 3 view, checkpoint có teacher) và `train_cdane` bên dưới là phần chung của các nhánh có Mean Teacher; ở notebook này chúng phục vụ §5.8.

In [ ]:
# === CELL-TAG: mt_helpers ===
# ══════════════════════════════════════════════════════════════════════════════
# MEAN TEACHER — hằng số, lịch ramp, loss, EMA
# Nguồn: github.com/CuriousAI/mean-teacher và github.com/milsever/plant-pathology
# ══════════════════════════════════════════════════════════════════════════════

EMA_DECAY     = 0.99   # paper Ilsever & Baz, lấy từ test "4000 labels/ResNet CIFAR-10"
CONS_MAX      = 9.0    # = C^2; paper chọn consistency weight trong [C, C^2], C=3
CONS_START    = 100    # epoch bắt đầu ramp (sau khi lambda_GRL đã gần bão hoà)
CONS_RAMP_LEN = 50     # số epoch để ramp tới CONS_MAX


def sigmoid_rampup(x, length):
    """Sigmoid ramp-up của Laine & Aila (ramps.py gốc).

    Args:
        x     : Số epoch đã trôi qua kể từ lúc bắt đầu ramp.
        length: Số epoch để đạt bão hoà. 0 nghĩa là không ramp.
    Returns:
        Hệ số trong (0, 1]. Tại x=0 trả về exp(-5) ~ 0.0067, không phải 0.
    """
    if length == 0:
        return 1.0
    x = float(np.clip(x, 0.0, length)) / length
    return float(np.exp(-5.0 * (1.0 - x) ** 2))


def consistency_weight(epoch, cons_max=CONS_MAX, start=CONS_START,
                       ramp_len=CONS_RAMP_LEN):
    """Trọng số w(t) của consistency loss. Đúng 0 trước `start`."""
    if epoch < start:
        return 0.0
    return cons_max * sigmoid_rampup(epoch - start, ramp_len)


def softmax_mse_loss(input_logits, target_logits):
    """MSE giữa hai phân phối softmax. Sao chép nguyên văn losses.py của repo gốc.

    Trả về TỔNG trên toàn batch (chia num_classes). Chỗ gọi phải chia batch_size,
    đúng như ft_meanteacher.py làm.
    """
    assert input_logits.size() == target_logits.size()
    input_softmax  = F.softmax(input_logits,  dim=1)
    target_softmax = F.softmax(target_logits, dim=1)
    num_classes    = input_logits.size()[1]
    return F.mse_loss(input_softmax, target_softmax, reduction='sum') / num_classes


def cdan_entropy_weight(probs):
    """Trọng số entropy của CDAN+E: w = 1 + exp(-H(p)).

    Sao chép nguyên văn định nghĩa trong cell 5.3 (CDAN+E). Cố ý lặp lại ở đây
    để section 5.5 chạy được mà KHÔNG phải chạy cell đó — nó là một run 300 epoch
    đầy đủ. Hai định nghĩa giống hệt nhau nên kết quả không đổi dù chạy thứ tự nào.

    Tin cậy (H thấp) -> ~2, mơ hồ -> ~1. Detach để làm hệ số, không phải đường
    truyền gradient.
    """
    ent = -(probs * torch.log(probs + 1e-5)).sum(dim=1)
    return (1.0 + torch.exp(-ent)).detach()


def create_teacher(student_phi, student_G, num_classes, device, backbone='resnet18'):
    """Dựng teacher khởi tạo giống hệt student và đã khoá gradient.

    pretrained=False để khỏi tải lại weight ImageNet — trọng số bị ghi đè ngay
    bằng state_dict của student. backbone phải khớp với student_phi (xem 5.7).

    Returns:
        (phi_t, G_t) — cả hai đã requires_grad_(False).
    """
    phi_t = FeatureExtractor(pretrained=False, backbone=backbone).to(device)
    G_t   = Classifier(phi_t.out_dim, num_classes).to(device)
    phi_t.load_state_dict(student_phi.state_dict())
    G_t.load_state_dict(student_G.state_dict())
    for p in list(phi_t.parameters()) + list(G_t.parameters()):
        p.requires_grad_(False)
    return phi_t, G_t


def update_ema(student_mods, teacher_mods, global_step, ema_decay=EMA_DECAY):
    """Cập nhật EMA. Sao chép update_ema_variables của CuriousAI/mean-teacher.

    Dùng true average cho tới khi exponential average đủ chính xác, nên alpha
    tăng dần từ 0 lên `ema_decay`.

    LƯU Ý: chỉ duyệt .parameters(), KHÔNG đụng buffer BatchNorm. Buffer của
    teacher được cập nhật nhờ chính forward pass của nó ở .train() mode.

    Returns:
        alpha đã dùng (để log).
    """
    alpha = min(1 - 1 / (global_step + 1), ema_decay)
    with torch.no_grad():
        for sm, tm in zip(student_mods, teacher_mods):
            for ema_p, p in zip(tm.parameters(), sm.parameters()):
                ema_p.data.mul_(alpha).add_(p.data, alpha=1 - alpha)
    return alpha


print('Mean Teacher helpers defined.')

Mean Teacher helpers defined.


In [ ]:
# === CELL-TAG: target_3view_ds ===
# ══════════════════════════════════════════════════════════════════════════════
# DATASET TARGET 3 ĐẦU RA — (plain, view1, view2)
#
# ĐÂY LÀ CHỖ QUYẾT ĐỊNH PHÉP SO SÁNH VỚI CELL 5.3 CÓ HỢP LỆ HAY KHÔNG.
#
# plain = VAL_TRANSFORM, tất định — ĐÚNG ảnh mà cell 5.3 đưa vào nhánh CDAN+E.
# view1/view2 = TRAIN_TRANSFORM, hai lần bốc mẫu độc lập — CHỈ dùng cho L_cons.
#
# ── Vì sao phải fork_rng ─────────────────────────────────────────────────────
# TRAIN_TRANSFORM (RandomCrop/Flip/ColorJitter) bốc số từ RNG TOÀN CỤC của
# PyTorch — đúng cái RNG mà augmentation của ảnh source cũng đang dùng. Vòng lặp
# là zip(train_loader_A_fbr, target_loader_3v), nên hai bên bốc xen kẽ nhau.
#
# Cell 5.3 KHÔNG bốc gì cho target (VAL_TRANSFORM tất định). Nếu ở đây ta bốc
# thêm 2 view mỗi ảnh mà không cô lập, thì từ batch thứ hai trở đi ảnh SOURCE
# của cell 5.5 đã khác ảnh source của cell 5.3 — cùng SEED cũng vô nghĩa, và
# chênh lệch accuracy không còn quy được về Mean Teacher.
#
# torch.random.fork_rng lưu lại state toàn cục, cho phép ta seed riêng cho phần
# augmentation, rồi trả state về nguyên vẹn. Dòng RNG mà ảnh source nhìn thấy vì
# thế giống hệt cell 5.3, từng bước một.
#
# ── Vì sao seed của view phụ thuộc (seed, epoch, idx) ────────────────────────
# Chỉ theo idx thì mọi epoch dùng lại đúng hai view cũ -> mất hết đa dạng
# augmentation. Có epoch thì mỗi epoch một cặp view mới, nhưng vẫn tái lập được,
# kể cả sau khi resume giữa chừng (khác với bộ đếm nội bộ, thứ sẽ reset về 0).
# ══════════════════════════════════════════════════════════════════════════════

class TargetThreeViewDataset(Dataset):
    """Target không nhãn, trả ((plain, view1, view2), -1).

    Args:
        samples        : List (path, label). Label bị vứt — luôn trả -1 để chặn
                         rò rỉ nhãn target vào quá trình huấn luyện UDA.
        plain_transform: Tất định (VAL_TRANSFORM). Ảnh cho nhánh CDAN+E.
        aug_transform  : Ngẫu nhiên (TRAIN_TRANSFORM). Hai view cho L_cons.
        base_seed      : Seed gốc; cùng base_seed + epoch -> cùng bộ view.
    """

    def __init__(self, samples, plain_transform, aug_transform, base_seed):
        self.samples         = samples
        self.plain_transform = plain_transform
        self.aug_transform   = aug_transform
        self.base_seed       = base_seed
        self.epoch           = 0

    def set_epoch(self, epoch):
        """Vòng lặp huấn luyện PHẢI gọi mỗi epoch, nếu không 300 epoch dùng
        chung đúng một cặp view và Mean Teacher gần như không học được gì."""
        self.epoch = epoch

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, _ = self.samples[idx]
        img = Image.open(path).convert('RGB')

        # Ngoài fork: dùng RNG toàn cục, nhưng VAL_TRANSFORM không bốc số nào.
        plain = self.plain_transform(img)

        # Trong fork: state toàn cục được cất đi rồi trả lại y nguyên.
        # devices=[] để khỏi fork luôn RNG của CUDA (chậm và không cần).
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(
                (self.base_seed * 1_000_003 + self.epoch * 10_007 + idx) % (2**31 - 1)
            )
            v1 = self.aug_transform(img)
            v2 = self.aug_transform(img)

        return (plain, v1, v2), -1


print('TargetThreeViewDataset defined.')


TargetThreeViewDataset defined.


In [ ]:
# === CELL-TAG: target_3view_loader ===
# ══════════════════════════════════════════════════════════════════════════════
# LOADER TARGET 3 VIEW
#
# batch_size/shuffle/drop_last giống hệt `target_loader` của cell 5.3, nên
# steps_per_epoch bằng nhau -> lịch lambda_GRL của hai cell trùng khít.
# ══════════════════════════════════════════════════════════════════════════════

# dataset_B_target là một Subset; lấy lại danh sách (path, label) gốc của nó.
_target_samples = [
    dataset_B_target.dataset.samples[i] for i in dataset_B_target.indices
]

dataset_B_target_3v = TargetThreeViewDataset(
    samples         = _target_samples,
    plain_transform = VAL_TRANSFORM,
    aug_transform   = TRAIN_TRANSFORM,
    base_seed       = SEED,
)

target_loader_3v = make_loader(
    dataset_B_target_3v, batch_size=64, shuffle=True, drop_last=True
)

assert len(target_loader_3v) == len(target_loader), (
    f'so batch lech voi cell 5.3: {len(target_loader_3v)} vs {len(target_loader)} '
    '-> lich lambda_GRL se khac nhau'
)

# ── Kiểm tra tại chỗ ─────────────────────────────────────────────────────────
(_plain, _v1, _v2), _lbl = next(iter(target_loader_3v))
assert _plain.shape == _v1.shape == _v2.shape == (64, 3, 224, 224), f'sai shape: {_plain.shape}'
assert not torch.allclose(_v1, _v2), \
    'view1 == view2 -> L_cons se luon = 0. Kiem tra TRAIN_TRANSFORM.'
assert not torch.allclose(_plain, _v1), \
    'plain == view1 -> plain dang bi augment. Kiem tra VAL_TRANSFORM.'
assert (_lbl == -1).all(), 'Target loader dang tra ve nhan that.'

# ── Kiểm tra fork_rng thật sự không đụng vào RNG toàn cục ───────────────────
# Đây là bất biến mà toàn bộ phép so sánh với cell 5.3 dựa vào, nên phải test.
torch.manual_seed(123)
_before = torch.rand(3)
torch.manual_seed(123)
dataset_B_target_3v.set_epoch(7)
_ = dataset_B_target_3v[0]          # sinh plain + 2 view augment
_after = torch.rand(3)
assert torch.equal(_before, _after), (
    'fork_rng bi ro ri: augmentation dang lam lech RNG toan cuc. '
    'Anh source cua cell 5.5 se khac cell 5.3.'
)

# ── Kiểm tra view đổi theo epoch và tái lập được ────────────────────────────
dataset_B_target_3v.set_epoch(1); (_, _e1a, _) = dataset_B_target_3v[0][0]
dataset_B_target_3v.set_epoch(2); (_, _e2,  _) = dataset_B_target_3v[0][0]
dataset_B_target_3v.set_epoch(1); (_, _e1b, _) = dataset_B_target_3v[0][0]
assert not torch.allclose(_e1a, _e2), 'view khong doi theo epoch -> mat da dang augment'
assert torch.allclose(_e1a, _e1b),    'cung epoch ma view khac nhau -> khong tai lap duoc'

print(f'Target 3-view loader ready: {len(dataset_B_target_3v)} anh, '
      f'{len(target_loader_3v)} batch/epoch (khop cell 5.3)')
print(f'  plain vs view1 : mean|.| = {(_plain - _v1).abs().mean():.4f}')
print(f'  view1 vs view2 : mean|.| = {(_v1 - _v2).abs().mean():.4f}')
print('  fork_rng khong ro ri sang RNG toan cuc  [OK]')
print('  view doi theo epoch, cung epoch thi tai lap  [OK]')
del _plain, _v1, _v2, _lbl, _before, _after, _e1a, _e1b, _e2


Target 3-view loader ready: 1038 anh, 16 batch/epoch (khop cell 5.3)
  plain vs view1 : mean|.| = 0.7369
  view1 vs view2 : mean|.| = 0.7642
  fork_rng khong ro ri sang RNG toan cuc  [OK]
  view doi theo epoch, cung epoch thi tai lap  [OK]


In [ ]:
# === CELL-TAG: mt_checkpoint ===
# ══════════════════════════════════════════════════════════════════════════════
# CHECKPOINT cho nhánh CDAN+E (+MT)
#
# Không sửa save_uda_checkpoint/load_uda_checkpoint của section 5.2 — cell CDAN+E
# một-seed vẫn đang dùng chúng. Đây là cặp riêng, có thêm state của teacher.
#
# 10 run x 300 epoch trên Colab chắc chắn sẽ bị ngắt session giữa chừng, nên
# resume là bắt buộc chứ không phải tuỳ chọn.
# ══════════════════════════════════════════════════════════════════════════════

def save_mt_checkpoint(path, epoch, step, best_loss, best_state,
                       phi, G, D, opt, sch, phi_t=None, G_t=None):
    """Lưu toàn bộ state để resume được.

    Args:
        epoch     : Epoch VỪA HOÀN THÀNH.
        step      : Bộ đếm iteration toàn cục (giữ liên tục lịch lambda và EMA).
        best_state: dict {'phi','G'} của model tốt nhất, hoặc None nếu chưa có.
        phi_t, G_t: Teacher; None với nhánh baseline.
    """
    ckpt = {
        'epoch': epoch, 'step': step,
        'best_loss': best_loss, 'best_state': best_state,
        'phi': phi.state_dict(), 'G': G.state_dict(), 'D': D.state_dict(),
        'optimizer': opt.state_dict(), 'scheduler': sch.state_dict(),
    }
    if phi_t is not None:
        # state_dict của teacher gồm cả BUFFER BatchNorm. Bắt buộc phải lưu:
        # chúng không được EMA mà tích luỹ dần qua forward pass, nên mất là
        # teacher quay về BN stats của ImageNet.
        ckpt['phi_t'] = phi_t.state_dict()
        ckpt['G_t']   = G_t.state_dict()
    torch.save(ckpt, path)


def load_mt_checkpoint(path, phi, G, D, opt, sch,
                       phi_t=None, G_t=None, device='cpu'):
    """Khôi phục state nếu checkpoint tồn tại.

    Returns:
        (start_epoch, step, best_loss, best_state).
        Nếu không có file: (1, 0, inf, None).
    """
    if not os.path.exists(path):
        return 1, 0, float('inf'), None

    ckpt = torch.load(path, map_location=device)
    phi.load_state_dict(ckpt['phi'])
    G.load_state_dict(ckpt['G'])
    D.load_state_dict(ckpt['D'])
    opt.load_state_dict(ckpt['optimizer'])
    sch.load_state_dict(ckpt['scheduler'])
    if phi_t is not None and 'phi_t' in ckpt:
        phi_t.load_state_dict(ckpt['phi_t'])
        G_t.load_state_dict(ckpt['G_t'])
    return ckpt['epoch'] + 1, ckpt['step'], ckpt['best_loss'], ckpt['best_state']


print('MT checkpoint helpers defined.')

MT checkpoint helpers defined.


### 5.2c — `train_cdane`

Vòng lặp CDAN+E (+ Mean Teacher). `target_ds` / `target_loader` / `test_loader` / `run_tag` là tham số tuỳ chọn để §5.8 truyền loader riêng (toàn bộ PPD); mặc định (None) dùng loader của A.

In [ ]:
# === CELL-TAG: train_cdane ===
# ══════════════════════════════════════════════════════════════════════════════
# CDAN+E + MEAN TEACHER — một seed, ghép cặp với cell 5.3
#
# Hàm này tái dựng ĐÚNG vòng lặp của cell 5.3 rồi cộng thêm w(t)*L_cons. Bốn
# điểm dưới đây là thứ giữ cho hai cell nhìn thấy cùng một dòng RNG; hỏng bất kỳ
# điểm nào là chênh lệch accuracy không còn quy về Mean Teacher được nữa.
#
# 1. THỨ TỰ KHỞI TẠO: FeatureExtractor -> Classifier -> CDANDisc, y như 5.3.
#    GradRevLayer không có tham số nên không tiêu thụ RNG, đặt đâu cũng được.
# 2. TEACHER DỰNG TRONG fork_rng: create_teacher() gọi FeatureExtractor +
#    Classifier, tức tiêu thụ RNG. Không fork thì toàn bộ augmentation ảnh
#    source sau đó lệch pha so với 5.3. (Trọng số random của teacher bị ghi đè
#    ngay bằng state_dict của student, nên fork không mất mát gì.)
# 3. g.manual_seed(seed * 1000 + epoch): công thức giống hệt cell 5.3.
# 4. set_epoch(epoch): view augment đổi theo epoch nhưng vẫn tái lập được, và
#    được sinh trong fork_rng nên không đụng RNG toàn cục.
#
# `use_mt=False` giữ lại KHÔNG phải để chạy baseline thật (baseline là cell 5.3)
# mà để kiểm chứng: chạy nó phải ra đúng con số của 5.3. Lệch là một trong bốn
# điểm trên đã hỏng.
# ══════════════════════════════════════════════════════════════════════════════

def train_cdane(seed=SEED, use_mt=True, max_epochs=300, min_sel=250, ckpt_dir=None,
                log_every=10, cons_max=CONS_MAX, cons_start=CONS_START,
                cons_ramp_len=CONS_RAMP_LEN, verbose=True, backbone='resnet18',
                target_ds=None, target_loader=None, test_loader=None, run_tag=None):
    """Huấn luyện CDAN+E + Mean Teacher trên một seed.

    Args:
        seed      : Phải bằng SEED để ghép cặp được với cell 5.3 (khi backbone
                    mặc định 'resnet18').
        use_mt    : True = CDAN+E + MT. False = tái hiện 5.3 (dùng để kiểm chứng).
        max_epochs: Tổng số epoch (paper: 300).
        min_sel   : Chỉ chọn model sau epoch này (paper: 250) — giống hệt 5.3.
        ckpt_dir  : Thư mục lưu checkpoint. None thì không lưu (dùng cho probe).
        backbone  : 'resnet18' (mặc định, khớp paper 10) hoặc 'resnet50' (xem
                    5.7). Đổi backbone là một thực nghiệm khác, KHÔNG nằm trong
                    phép so sánh RNG-pairing giữa 5.3/5.5 — rng_probe.py chỉ
                    kiểm chứng cho backbone='resnet18'.
        target_ds, target_loader, test_loader:
                    Mặc định (None) = dataset_B_target_3v / target_loader_3v /
                    test_loader_B như mọi cell 5.3-5.7. Chỉ mục 5.8 (giao thức
                    MSUN) truyền vào loader khác.
        run_tag   : Ghi đè tag dùng trong tên file checkpoint, để run của 5.8
                    không đè lên checkpoint của 5.5/5.7.

    Returns:
        dict: seed, use_mt, backbone, acc_selected (%), acc_student_final (%),
              best_val_loss, epochs_run.
    """
    tag = ('mt' if use_mt else 'base') + ('' if backbone == 'resnet18' else f'_{backbone}')
    if run_tag is not None:
        tag = run_tag
    if target_ds is None:
        target_ds = dataset_B_target_3v
    if target_loader is None:
        target_loader = target_loader_3v
    if test_loader is None:
        test_loader = test_loader_B
    set_seed(seed)

    # ── (1) Thứ tự khởi tạo y hệt cell 5.3 ───────────────────────────────────
    phi = FeatureExtractor(pretrained=True, backbone=backbone).to(DEVICE)
    G   = Classifier(phi.out_dim, NUM_CLASSES).to(DEVICE)
    D   = CDANDisc(phi.out_dim, NUM_CLASSES).to(DEVICE)
    grl = GradRevLayer().to(DEVICE)

    # ── (2) Teacher dựng trong fork_rng ──────────────────────────────────────
    with torch.random.fork_rng(devices=[]):
        phi_t, G_t = create_teacher(phi, G, NUM_CLASSES, DEVICE, backbone=backbone)

    opt = optim.AdamW(
        list(phi.parameters()) + list(G.parameters()) + list(D.parameters()),
        lr=1e-3, weight_decay=0.01
    )
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

    ce_mean   = nn.CrossEntropyLoss()
    ce_persam = nn.CrossEntropyLoss(reduction='none')

    ckpt_path = best_path = None
    start_epoch, step, best_loss, best_state = 1, 0, float('inf'), None
    if ckpt_dir is not None:
        ckpt_path = os.path.join(ckpt_dir, f'cdane_{tag}_s{seed}_ckpt.pth')
        best_path = os.path.join(ckpt_dir, f'cdane_{tag}_s{seed}_best.pth')
        start_epoch, step, best_loss, best_state = load_mt_checkpoint(
            ckpt_path, phi, G, D, opt, sch, phi_t, G_t, device=DEVICE
        )
        if start_epoch > 1 and verbose:
            print(f'  [resume] tiep tuc tu epoch {start_epoch}')

    steps_per_epoch = min(len(train_loader_A_fbr), len(target_loader))
    total_steps     = max_epochs * steps_per_epoch
    sel_phi, sel_G  = (phi_t, G_t) if use_mt else (phi, G)
    lam             = 0.0

    for epoch in range(start_epoch, max_epochs + 1):
        g.manual_seed(seed * 1000 + epoch)          # (3) khop cell 5.3
        target_ds.set_epoch(epoch)                  # (4) view moi moi epoch

        phi.train(); G.train(); D.train()
        phi_t.train(); G_t.train()      # BẮT BUỘC train(): BN buffer của teacher
                                        # không được EMA, chỉ cập nhật qua forward
        w_cons = consistency_weight(epoch, cons_max, cons_start, cons_ramp_len)

        ep_cls = ep_cdan = ep_cons = 0.0
        tgt_hist = torch.zeros(NUM_CLASSES, dtype=torch.long)

        for (si, sl), ((t_plain, t_v1, t_v2), _) in zip(train_loader_A_fbr,
                                                        target_loader):
            si, sl  = si.to(DEVICE), sl.to(DEVICE)
            t_plain = t_plain.to(DEVICE)

            lam = get_lambda(step, total_steps)
            grl.set_lam(lam)
            step += 1
            opt.zero_grad()

            # ── Nhánh CDAN+E: sao chép nguyên văn cell 5.3 ──────────────────
            # Target là ẢNH KHÔNG AUGMENT (t_plain) — đúng input của 5.3.
            sf = phi(si)
            tf = phi(t_plain)
            logits_s = G(sf)
            logits_t = G(tf)
            sp = torch.softmax(logits_s, dim=1)
            tp = torch.softmax(logits_t, dim=1)

            L_cls = ce_mean(logits_s, sl)

            # Detach PREDICTION (đúng CDAN gốc) nhưng giữ gradient của FEATURE,
            # để tín hiệu đối kháng còn tới được phi qua GRL.
            src_mm = grl(multilinear_map(sf, sp.detach()))
            tgt_mm = grl(multilinear_map(tf, tp.detach()))

            sw = cdan_entropy_weight(sp); sw = sw / sw.sum()
            tw = cdan_entropy_weight(tp); tw = tw / tw.sum()
            src_d = (sw * ce_persam(D(src_mm), domain_lbl(si.size(0),      True,  DEVICE))).sum()
            tgt_d = (tw * ce_persam(D(tgt_mm), domain_lbl(t_plain.size(0), False, DEVICE))).sum()
            L_cdan = (src_d + tgt_d) / 2.0

            # ── Nhánh Mean Teacher: forward RIÊNG trên hai view augment ─────
            # Tách hẳn khỏi nhánh CDAN, nên L_cdan không đổi một chữ so với 5.3.
            # Giá phải trả: thêm 2 forward pass mỗi iteration.
            if use_mt:
                t_v1, t_v2 = t_v1.to(DEVICE), t_v2.to(DEVICE)
                student_logits = G(phi(t_v1))
                with torch.no_grad():
                    teacher_logits = G_t(phi_t(t_v2))
                L_cons = softmax_mse_loss(student_logits, teacher_logits) / t_v1.size(0)
                probe = teacher_logits
            else:
                # Không có update_ema thì teacher đứng nguyên tại init: backbone
                # ImageNet + head ngẫu nhiên. Dự đoán vô nghĩa -> đo bằng student.
                L_cons = torch.zeros((), device=DEVICE)
                probe = logits_t.detach()

            tgt_hist += torch.bincount(probe.argmax(1).cpu(), minlength=NUM_CLASSES)

            total = L_cls + L_cdan + (w_cons * L_cons if use_mt else 0.0)
            total.backward()
            opt.step()

            if use_mt:
                update_ema([phi, G], [phi_t, G_t], step)

            ep_cls  += L_cls.item()
            ep_cdan += L_cdan.item()
            ep_cons += L_cons.item()

        sch.step()

        # ── Validation trên FBR val — cùng tiêu chí với cell 5.3 ────────────
        sel_phi.eval(); sel_G.eval()
        vl, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for vi, vlab in val_loader_A_fbr:
                vi, vlab = vi.to(DEVICE), vlab.to(DEVICE)
                logits = sel_G(sel_phi(vi))
                vl += ce_mean(logits, vlab).item()
                vc += (logits.argmax(1) == vlab).sum().item()
                vt += vlab.size(0)
        val_loss, val_acc = vl / len(val_loader_A_fbr), vc / vt

        if epoch >= min_sel and val_loss < best_loss:
            best_loss  = val_loss
            best_state = {'phi': copy.deepcopy(sel_phi.state_dict()),
                          'G':   copy.deepcopy(sel_G.state_dict())}
            if best_path:
                torch.save(best_state, best_path)

        if verbose and (epoch % log_every == 0 or epoch == max_epochs):
            share = (tgt_hist.float() / max(tgt_hist.sum().item(), 1) * 100)
            print(f'  [{tag}] ep {epoch:03d}/{max_epochs} '
                  f'val_acc={val_acc*100:5.1f}% lam={lam:.3f} w={w_cons:.2f} '
                  f'| cls={ep_cls/steps_per_epoch:.3f} '
                  f'cdan={ep_cdan/steps_per_epoch:.3f} '
                  f'cons={ep_cons/steps_per_epoch:.4f} '
                  f'| tgt%=[{share[0]:.0f},{share[1]:.0f},{share[2]:.0f}]')
            if share.max().item() > 70.0:
                who = 'teacher' if use_mt else 'student'
                print(f'     CANH BAO: {who} gan sap ve 1 lop '
                      f'({share.max():.0f}%) - nghi ngo class collapse')

        if ckpt_path and (epoch % log_every == 0 or epoch == max_epochs):
            save_mt_checkpoint(ckpt_path, epoch, step, best_loss, best_state,
                               phi, G, D, opt, sch, phi_t, G_t)

    # ── Đánh giá ─────────────────────────────────────────────────────────────
    if best_state is None:
        print(f'  CANH BAO [{tag}]: chua toi cua so chon model, dung weight cuoi.')
        best_state = {'phi': copy.deepcopy(sel_phi.state_dict()),
                      'G':   copy.deepcopy(sel_G.state_dict())}

    eval_phi = FeatureExtractor(pretrained=False, backbone=backbone).to(DEVICE)
    eval_G   = Classifier(eval_phi.out_dim, NUM_CLASSES).to(DEVICE)
    eval_phi.load_state_dict(best_state['phi'])
    eval_G.load_state_dict(best_state['G'])

    acc_selected      = uda_test(eval_phi, eval_G, test_loader, DEVICE) * 100
    acc_student_final = uda_test(phi, G, test_loader, DEVICE) * 100

    if use_mt and acc_student_final > acc_selected + 3.0:
        print(f'  CANH BAO: student ({acc_student_final:.2f}%) hon han teacher '
              f'({acc_selected:.2f}%) - nghi ngo BN buffer chua sync')

    return {
        'seed': seed, 'use_mt': use_mt, 'backbone': backbone,
        'acc_selected': acc_selected,
        'acc_student_final': acc_student_final,
        'best_val_loss': best_loss,
        'epochs_run': max_epochs,
    }


print('train_cdane() defined.  Mac dinh: seed=SEED, use_mt=True.')


train_cdane() defined.  Mac dinh: seed=SEED, use_mt=True.


### 5.8 — So sánh với CBPUDA (Huang & Tsai) theo giao thức của họ

> Huang L.-C., Tsai H.-H. *Contrastive Bi-Projector for Unsupervised Domain Adaption.* arXiv:2308.07017.

Table IV của CBPUDA (PV → PP, táo, 3 lớp, ResNet-50) gồm 7 phương pháp. Sáu dòng baseline (ResNet-50, DAN, DANN,
DSAN, MRAN) trùng **từng số** với Table 6 của MSUN, nên nhiều khả năng bài chép lại từ MSUN chứ không tự chạy.
Hai dòng do chính CBPUDA báo cáo là BCDM và CBPUDA. Thứ tự cột của Table IV là healthy / rust / scab, ngược với MSUN.

**Giới hạn phải nêu khi viết:**

1. **Cùng dữ liệu, khác phương pháp.** CBPUDA không dùng FBR; pipeline này có FBR. Chênh lệch gộp cả FBR lẫn
   CDAN+E/MT. Muốn tách ra thì cần thêm dòng CDAN+E không FBR (chưa chạy).
2. **Khác optimizer và ngân sách huấn luyện** (xem bảng ở đầu notebook). CBPUDA không công bố code.
3. **Cột AVG của Table IV gần với trung bình 3 lớp** (CBPUDA: trung bình 3 lớp 94.89, bảng ghi 94.79), trong khi
   các dòng chép từ MSUN là accuracy tổng. Ta in cả `mean3` lẫn `acc` cho mọi dòng.
4. **Chọn model:** với `CBP_TRAIN_ON_ALL_SOURCE=True`, model được chọn theo loss trên chính tập train source
   (không dùng nhãn target), tương đương "model cuối" của bài báo.
5. Một seed (`SEED`).


In [ ]:
# === CELL-TAG: cbp_setup ===
# ══════════════════════════════════════════════════════════════════════════════
# GIAO THỨC CBPUDA: source = toàn bộ PV táo, target = đánh giá = toàn bộ PPD single-label
# ══════════════════════════════════════════════════════════════════════════════

CBP_EPOCHS  = 300   # giu nhu A/B de so voi B chi khac o du lieu source
CBP_MIN_SEL = 250

# (use_mt, backbone): mặc định hai run ResNet-50 để đặt cạnh Table IV của CBPUDA.
CBP_RUNS = [(True, 'resnet50')]

dataset_cbp_3v = TargetThreeViewDataset(
    samples         = ppd_samples,          # TOÀN BỘ PPD, nhãn bị vứt (trả -1)
    plain_transform = VAL_TRANSFORM,
    aug_transform   = TRAIN_TRANSFORM,
    base_seed       = SEED,
)
target_loader_cbp = make_loader(dataset_cbp_3v, batch_size=64, shuffle=True, drop_last=True)
eval_loader_cbp   = make_loader(
    PlantDiseaseDataset(samples=ppd_samples, classes=CLASS_NAMES,
                        transform=VAL_TRANSFORM, labeled=True),
    batch_size=64, shuffle=False,
)

assert len(pvd_samples) == sum(CBP_PV_COUNTS.values()), 'source khong phai 2550 anh'
assert len(ppd_samples) == sum(CBP_PP_COUNTS.values()), 'target khong phai 1730 anh'
assert set(fbr_bg_paths) <= {p for p, _ in ppd_samples}, 'anh nen FBR nam ngoai PPD'

# Table IV cua CBPUDA (PV -> PP). Thu tu cot theo bai: healthy, rust, scab, AVG.
CBP_TABLE4 = {
    #  method              : (healthy, rust, scab, avg)
    'ResNet-50 (w/o DA)'   : (11.02, 24.10, 79.54, 39.27),
    'DAN'                  : (11.84, 34.99, 89.02, 48.89),
    'DANN'                 : (28.71, 30.26, 85.12, 50.04),
    'DSAN'                 : (51.24, 35.42, 83.96, 56.41),
    'MRAN'                 : (70.87, 52.57, 81.28, 67.45),
    'BCDM'                 : (97.09,  1.93, 88.51, 59.65),
    'CBPUDA'               : (94.96, 92.92, 96.79, 94.79),
}


def eval_per_class(phi, G, loader, device):
    # Tra ve (acc tong %, {lop: acc %}, confusion [true][pred]).
    phi.eval(); G.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = G(phi(imgs.to(device))).argmax(1).cpu().numpy()
            for t, p in zip(lbls.numpy(), preds):
                cm[t, p] += 1
    per_class = {c: 100.0 * cm[i, i] / max(cm[i].sum(), 1) for i, c in enumerate(CLASS_NAMES)}
    return 100.0 * np.trace(cm) / cm.sum(), per_class, cm.tolist()


print(f'Giao thuc CBPUDA: source = {len(pvd_samples)} anh PV (train {len(fbr_tr_idx)}), '
      f'target = danh gia = {len(ppd_samples)} anh PPD')
print(f'  target loader: {len(target_loader_cbp)} batch | source train: {len(train_loader_A_fbr)} batch '
      f'-> {min(len(train_loader_A_fbr), len(target_loader_cbp))} buoc/epoch')
print(f'  runs: {CBP_RUNS}')


Giao thuc CBPUDA: source = 2550 anh PV (train 2550), target = danh gia = 1730 anh PPD
  target loader: 27 batch | source train: 39 batch -> 27 buoc/epoch
  runs: [(True, 'resnet50')]


In [21]:
# === CELL-TAG: run_cbp ===
# ══════════════════════════════════════════════════════════════════════════════
# CHẠY THEO GIAO THỨC CBPUDA — mỗi (use_mt, backbone) một JSON, có rồi thì bỏ qua.
# Checkpoint mỗi 10 epoch, resume tự động (tag riêng 'cbp_*', không đè run của A/B).
# ══════════════════════════════════════════════════════════════════════════════

import json as _json

if 'all_results' not in globals():
    all_results = {}


def _cbp_tag(use_mt, backbone):
    # Ten file phan biet hai che do chon model, de doi cong tac khong bo qua nham ket qua cu.
    split = 'all' if CBP_TRAIN_ON_ALL_SOURCE else 'split'
    return f'cbp_{split}_{"mt" if use_mt else "base"}_{backbone}'


cbp_results = {}
for _use_mt, _bb in CBP_RUNS:
    _tag  = _cbp_tag(_use_mt, _bb)
    _json_path = os.path.join(RUN_DIR, f'cdane_{_tag}_s{SEED}_result.json')

    if os.path.exists(_json_path):
        with open(_json_path) as f:
            _res = _json.load(f)
        print(f'[skip] da co {os.path.basename(_json_path)}')
    else:
        print('=' * 60)
        print(f'CBPUDA protocol: {"CDAN+E + MT" if _use_mt else "CDAN+E"}, {_bb}  (seed={SEED})')
        print('=' * 60)
        _res = train_cdane(seed=SEED, use_mt=_use_mt, max_epochs=CBP_EPOCHS, min_sel=CBP_MIN_SEL,
                           ckpt_dir=RUN_DIR, backbone=_bb, run_tag=_tag,
                           target_ds=dataset_cbp_3v, target_loader=target_loader_cbp,
                           test_loader=eval_loader_cbp)

        # Accuracy theo lớp của model đã chọn (train_cdane chỉ trả accuracy tổng).
        _best_path = os.path.join(RUN_DIR, f'cdane_{_tag}_s{SEED}_best.pth')
        if os.path.exists(_best_path):
            _state = torch.load(_best_path, map_location=DEVICE)
            _phi = FeatureExtractor(pretrained=False, backbone=_bb).to(DEVICE)
            _G   = Classifier(_phi.out_dim, NUM_CLASSES).to(DEVICE)
            _phi.load_state_dict(_state['phi']); _G.load_state_dict(_state['G'])
            _acc, _pc, _cm = eval_per_class(_phi, _G, eval_loader_cbp, DEVICE)
            assert abs(_acc - _res['acc_selected']) < 1e-6, 'per-class eval lech voi train_cdane'
            _res.update({'per_class': _pc, 'confusion': _cm})
            del _phi, _G, _state
        _res['protocol'] = 'cbpuda_transductive_full_pv_full_ppd'
        _res['train_on_all_source'] = CBP_TRAIN_ON_ALL_SOURCE
        with open(_json_path, 'w') as f:
            _json.dump(_res, f, indent=2)

    cbp_results[(_use_mt, _bb)] = _res
    _name = f'{"CDAN+E + MT" if _use_mt else "CDAN+E"} ({_bb}, CBPUDA protocol)'
    all_results[_name] = _res['acc_selected']
    _pc = _res.get('per_class', {})
    print(f'-> {_name}: {_res["acc_selected"]:.2f}%  '
          + ', '.join(f'{c}={v:.2f}%' for c, v in _pc.items()))


CBPUDA protocol: CDAN+E + MT, resnet50  (seed=42)
  [cbp_all_mt_resnet50] ep 010/300 val_acc= 93.7% lam=0.165 w=0.00 | cls=0.067 cdan=0.639 cons=0.0568 | tgt%=[58,11,30]
  [cbp_all_mt_resnet50] ep 020/300 val_acc= 90.6% lam=0.321 w=0.00 | cls=0.096 cdan=0.670 cons=0.0495 | tgt%=[55,15,31]
  [cbp_all_mt_resnet50] ep 030/300 val_acc= 94.3% lam=0.462 w=0.00 | cls=0.065 cdan=0.652 cons=0.0544 | tgt%=[54,12,34]
  [cbp_all_mt_resnet50] ep 040/300 val_acc= 92.5% lam=0.582 w=0.00 | cls=0.053 cdan=0.649 cons=0.0428 | tgt%=[51,18,31]
  [cbp_all_mt_resnet50] ep 050/300 val_acc= 88.1% lam=0.682 w=0.00 | cls=0.072 cdan=0.661 cons=0.0672 | tgt%=[52,19,29]
  [cbp_all_mt_resnet50] ep 060/300 val_acc= 89.5% lam=0.761 w=0.00 | cls=0.027 cdan=0.679 cons=0.0465 | tgt%=[53,20,27]
  [cbp_all_mt_resnet50] ep 070/300 val_acc= 87.1% lam=0.823 w=0.00 | cls=0.036 cdan=0.685 cons=0.0560 | tgt%=[56,14,30]
  [cbp_all_mt_resnet50] ep 080/300 val_acc= 85.7% lam=0.870 w=0.00 | cls=0.033 cdan=0.649 cons=0.0562 | tgt%=[

In [22]:
# === CELL-TAG: compare_cbp ===
# ══════════════════════════════════════════════════════════════════════════════
# ĐẶT CẠNH TABLE IV CỦA CBPUDA (PV -> PP, táo, 3 lớp)
# ══════════════════════════════════════════════════════════════════════════════

assert cbp_results, 'Chua co ket qua. Chay cell run_cbp truoc.'

_order = ('healthy', 'rust', 'scab')          # thu tu cot cua Table IV
print('=' * 76)
print(f'{"Method":<36}{"healthy":>9}{"rust":>9}{"scab":>9}{"mean3":>7}{"acc":>8}')
print('-' * 76)
for _m, (_h, _r, _s, _avg) in CBP_TABLE4.items():
    print(f'{_m + " [CBP T4]":<36}{_h:>9.2f}{_r:>9.2f}{_s:>9.2f}{(_h + _r + _s) / 3:>7.2f}{_avg:>8.2f}')
print('-' * 76)
for (_use_mt, _bb), _res in cbp_results.items():
    _name = f'{"CDAN+E + MT" if _use_mt else "CDAN+E"} ({_bb}) [ours]'
    _pc = _res.get('per_class')
    if _pc:
        _cols = [_pc[c] for c in _order]
        print(f'{_name:<36}' + ''.join(f'{v:>9.2f}' for v in _cols)
              + f'{np.mean(_cols):>7.2f}{_res["acc_selected"]:>8.2f}')
    else:
        print(f'{_name:<36}{"—":>9}{"—":>9}{"—":>9}{"—":>7}{_res["acc_selected"]:>8.2f}')
print('=' * 76)
print('acc   : CBPUDA = cot "AVG" cua Table IV; ours = accuracy tong tren 1730 anh PPD.')
print('mean3 : trung binh 3 lop, tu tinh cho moi dong de so cung mot dinh nghia.')
print()
print('LUU Y KHI PHAT BIEU')
print('-' * 76)
print('  1. Cung du lieu (2550 PV -> 1730 PP, transductive) va cung backbone ResNet-50;')
print('     KHAC: ta co FBR, khac optimizer / so iteration. CBPUDA khong cong bo code.')
print('  2. Cac dong ResNet-50/DAN/DANN/DSAN/MRAN trung tung so voi Table 6 cua MSUN')
print('     (chep lai, dung source 1107 anh cua MSUN), chi BCDM va CBPUDA la cua bai nay.')
print(f'  3. n=1 (seed {SEED}). Chenh lech vai diem co the la nhieu seed.')


Method                                healthy     rust     scab  mean3     acc
----------------------------------------------------------------------------
ResNet-50 (w/o DA) [CBP T4]             11.02    24.10    79.54  38.22   39.27
DAN [CBP T4]                            11.84    34.99    89.02  45.28   48.89
DANN [CBP T4]                           28.71    30.26    85.12  48.03   50.04
DSAN [CBP T4]                           51.24    35.42    83.96  56.87   56.41
MRAN [CBP T4]                           70.87    52.57    81.28  68.24   67.45
BCDM [CBP T4]                           97.09     1.93    88.51  62.51   59.65
CBPUDA [CBP T4]                         94.96    92.92    96.79  94.89   94.79
----------------------------------------------------------------------------
CDAN+E + MT (resnet50) [ours]           99.42    48.71    89.70  79.28   77.86
acc   : CBPUDA = cot "AVG" cua Table IV; ours = accuracy tong tren 1730 anh PPD.
mean3 : trung binh 3 lop, tu tinh cho moi dong de so c